# Cyberpunk City Text-to-Image Generation Prototype

This notebook is a complete, beginner-friendly prototype for generating cyberpunk city images using a **pretrained Stable Diffusion** model from the `diffusers` library.

You will learn how to:
- Load a pretrained model
- Enter custom prompts
- Use negative prompts
- Control guidance scale and inference steps
- Set seeds for reproducible results
- Save generated outputs
- Run a small prompt engineering experiment


## 1) Install and import dependencies

> If you are running this notebook for the first time, uncomment the install line in the next cell.

In [ ]:
# !pip install -q diffusers transformers accelerate safetensors torch pillow

import os
from pathlib import Path
from datetime import datetime

import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
from IPython.display import display


## 2) Configuration

Set model and output settings in one place so they are easy to modify during experiments.

In [ ]:
# You can switch to another Stable Diffusion model if desired
MODEL_ID = "runwayml/stable-diffusion-v1-5"

# Notebook is inside /notebooks, so use ../outputs for saved images
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Hardware selection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Model: {MODEL_ID}")
print(f"Device: {DEVICE}")
print(f"Torch dtype: {DTYPE}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 3) Load Stable Diffusion pipeline

The first run may take time because model weights are downloaded and cached.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe = pipe.to(DEVICE)

# Optional speed/memory optimization
if DEVICE == "cuda":
    pipe.enable_attention_slicing()

print("Pipeline loaded successfully.")

## 4) Core generation function

This helper function encapsulates all key generation controls:
- **prompt** (what to generate)
- **negative_prompt** (what to avoid)
- **guidance_scale** (how strongly to follow text)
- **num_inference_steps** (quality/speed trade-off)
- **seed** (reproducibility)
- save image to `../outputs`


In [ ]:
def generate_cyberpunk_image(
    prompt: str,
    negative_prompt: str = "",
    guidance_scale: float = 7.5,
    num_inference_steps: int = 30,
    height: int = 512,
    width: int = 512,
    seed: int | None = None,
    file_prefix: str = "cyberpunk_notebook",
):
    if not prompt or not prompt.strip():
        raise ValueError("Prompt cannot be empty.")

    generator = None
    if seed is not None:
        generator = torch.Generator(device=DEVICE).manual_seed(seed)

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        height=height,
        width=width,
        generator=generator,
    )

    image = result.images[0]

    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S_%f")
    output_path = OUTPUT_DIR / f"{file_prefix}_{timestamp}.png"
    image.save(output_path)

    return image, output_path


## 5) Single-image generation (custom prompt input)

Edit the values in the next cell to generate your own cyberpunk city image.

In [ ]:
# ===== User-editable inputs =====
prompt = "A rainy cyberpunk megacity at night, neon signs in Japanese, flying cars, volumetric lighting, ultra-detailed"
negative_prompt = "blurry, low quality, washed out colors, distorted buildings, extra limbs, watermark, text artifacts"
guidance_scale = 8.0
num_inference_steps = 35
seed = 42
height = 512
width = 512

image, saved_path = generate_cyberpunk_image(
    prompt=prompt,
    negative_prompt=negative_prompt,
    guidance_scale=guidance_scale,
    num_inference_steps=num_inference_steps,
    height=height,
    width=width,
    seed=seed,
    file_prefix="single_run",
)

display(image)
print(f"Saved image: {saved_path}")

## 6) Quick parameter intuition (beginner guide)

- **Guidance scale**
  - Lower (e.g., 5–6): more creative freedom, sometimes less prompt adherence.
  - Higher (e.g., 8–12): follows prompt more strongly, but can look over-constrained.
- **Inference steps**
  - Fewer steps (e.g., 20): faster, sometimes lower detail.
  - More steps (e.g., 40+): slower, usually better detail up to a point.
- **Seed**
  - Same prompt + same seed + same settings => reproducible output.
  - Change seed to explore different compositions.


## 7) Prompt Engineering Experiment Section

This section compares multiple prompt styles while keeping technical settings controlled.

### Experiment design
- Constant: model, guidance scale, steps, resolution
- Variable: prompt wording
- Objective: observe how descriptive detail changes scene composition and aesthetics


In [ ]:
experiment_prompts = [
    "Cyberpunk city skyline at night, neon rain, cinematic wide shot",
    "Street-level view of a cyberpunk alley, neon reflections on wet asphalt, fog, photorealistic",
    "Futuristic cyberpunk market district, holograms, crowds, synthwave color palette, highly detailed concept art",
    "Aerial drone view of a dense cyberpunk megacity, giant holographic ads, moody atmosphere, 8k",
]

experiment_negative = "blurry, low resolution, bad perspective, deformed structures, text, watermark"
experiment_guidance = 7.5
experiment_steps = 30
experiment_seed = 123

experiment_results = []

for i, p in enumerate(experiment_prompts, start=1):
    img, path = generate_cyberpunk_image(
        prompt=p,
        negative_prompt=experiment_negative,
        guidance_scale=experiment_guidance,
        num_inference_steps=experiment_steps,
        seed=experiment_seed,
        file_prefix=f"prompt_exp_{i}",
    )
    experiment_results.append((p, path, img))
    print(f"[{i}] Saved: {path}")

In [ ]:
# Display experiment outputs one by one with prompt text
for i, (p, path, img) in enumerate(experiment_results, start=1):
    print(f"\n=== Experiment {i} ===")
    print(f"Prompt: {p}")
    print(f"File: {path}")
    display(img)

## 8) Seed variation mini-experiment

Use the same prompt but different seeds to explore composition diversity.

In [ ]:
base_prompt = "A neon-lit cyberpunk boulevard with towering skyscrapers and flying vehicles, rainy night"
base_negative = "blurry, low quality, artifacts, watermark"
seeds = [7, 21, 42]

seed_results = []
for s in seeds:
    img, path = generate_cyberpunk_image(
        prompt=base_prompt,
        negative_prompt=base_negative,
        guidance_scale=8.0,
        num_inference_steps=32,
        seed=s,
        file_prefix=f"seed_{s}",
    )
    seed_results.append((s, path, img))

for s, path, img in seed_results:
    print(f"Seed {s} -> {path}")
    display(img)

## 9) Suggested team tasks (for course project execution)

- **Member 1 (Model):** Compare SD v1.5 vs another checkpoint and report quality/speed.
- **Member 2 (Prompt Engineering):** Build reusable prompt templates and negative prompt library.
- **Member 3 (Evaluation):** Design visual scoring rubric and collect user preference feedback.
- **Member 4 (Backend):** Integrate best notebook settings into Flask app defaults.
- **Member 5 (Frontend):** Build UI controls for advanced parameters and output gallery.
- **Member 6 (Docs/Presentation):** Prepare report, ablation visuals, and final demo narrative.


## 10) Troubleshooting

- **Out-of-memory errors on GPU**: reduce `height/width` to 512 or lower, reduce batch size (1), close other GPU programs.
- **Very slow generation**: use GPU runtime if available; CPU inference can be slow.
- **Poor quality outputs**: refine prompt detail, improve negative prompt, adjust steps/guidance.
- **Reproducibility mismatch**: keep model, seed, steps, guidance, and resolution fixed.
